# LIMPIEZA DE DATOS — ENTIDADES DE DEPÓSITO
### Balance agregado: Activo (be0451) y Pasivo (be0452)
### Incluye:
- Importación de librerías
- Carga de datasets (formato Boletín Estadístico BdE)
- Conversión de tipos y limpieza
- Conversión de unidades: miles de euros → millones de euros
- Verificación MECE de consistencia entre totales y componentes
- Análisis de cobertura de valores nulos
- Visualización de series principales
- Exportación de archivos finales

---
**Fuentes (Banco de España — Boletín Estadístico, capítulo 4):**
- `be0451.csv` — *ED. Balance componentes. Activo.* Cobertura: enero 1962 – marzo 2026. Mensual. 13 series (12 utilizadas + 1 discontinuada descartada).
- `be0452.csv` — *ED. Balance componentes. Pasivo.* Cobertura: enero 1962 – marzo 2026. Mensual. 10 series.

# 0. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# 1. Constantes globales

In [2]:
RUTA = '../Fuentes/BDE/entidades_deposito/'

# Archivos fuente
ARCHIVO_ACTIVO = RUTA + 'Entidades de Deposito Activo Historico, be0451.csv'
ARCHIVO_PASIVO = RUTA + 'Entidades de Deposito Pasivo Historico, be0452.csv'

# Los archivos BdE tienen 6 filas de metadatos antes de la cabecera
FILAS_METADATA_BDE = 6

# Conversión de unidades: miles de euros → millones de euros
# Para mantener coherencia con los archivos PIB del proyecto (en M EUR)
FACTOR_MILES_A_MILLONES = 1 / 1000

print(f'Factor conversión: {FACTOR_MILES_A_MILLONES} (miles EUR → M EUR)')

Factor conversión: 0.001 (miles EUR → M EUR)


# 2. Carga de datasets
### 2.1 Exploración de metadatos
Las primeras 6 filas contienen información sobre las series (código, alias, descripción, unidades, frecuencia).

In [3]:
# Leer metadatos de ambos archivos para documentar las series
for nombre, archivo in [('ACTIVO (be0451)', ARCHIVO_ACTIVO), ('PASIVO (be0452)', ARCHIVO_PASIVO)]:
    meta = pd.read_csv(archivo, nrows=FILAS_METADATA_BDE, header=None, encoding='latin1')
    print(f'=== {nombre} ===')
    etiquetas = ['CÓDIGO', 'NÚM. SECUENCIAL', 'ALIAS', 'DESCRIPCIÓN', 'UNIDADES', 'FRECUENCIA']
    for i, (etiq, fila) in enumerate(zip(etiquetas, meta.values)):
        print(f'  {etiq}: {fila[0]}')
        # Mostrar descripciones de las series
        if etiq == 'DESCRIPCIÓN':
            for j, desc in enumerate(fila[1:], 1):
                alias = meta.iloc[2, j]
                print(f'    [{alias}] {desc}')
    print()

=== ACTIVO (be0451) ===
  CÓDIGO: CÓDIGO DE LA SERIE
  NÚM. SECUENCIAL: NÚMERO SECUENCIAL
  ALIAS: ALIAS DE LA SERIE
  DESCRIPCIÓN: DESCRIPCIÓN DE LA SERIE
    [BE_4_51.1] ED. Balance componentes. Activo. Total
    [BE_4_51.2] ED. Balance componentes. Activo. Créditos. Residentes en España. Sistema crediticio
    [BE_4_51.3] ED. Créditos. A AAPP
    [BE_4_51.4] ED. Créditos. A OSR
    [BE_4_51.5] ED. Balance componentes. Activo. Créditos. Resto del mundo
    [BE_4_51.6] ED. Balance componentes. Activo. Valores distintos de acciones y participaciones. Residentes en España
    [BE_4_51.7] ED. Balance componentes. Activo. Valores distintos de acciones y participaciones. Resto del mundo
    [BE_4_51.8] ED. Balance componentes. Activo. acciones y participaciones. Residentes en España
    [BE_4_51.9] ED. Balance componentes. Activo. acciones y participaciones. Resto del mundo
    [BE_4_51.10] ED. Balance componentes. Activo. Activos no sectorizados. Efectivo
    [BE_4_51.11] ED. Balance comp

### 2.2 Carga de datos

In [4]:
def cargar_bde_csv(filepath, nombre, drop_cols=None):
    """
    Carga un CSV del Boletín Estadístico del BdE (formato be04).
    - Salta las 6 filas de metadatos y usa la fila 7 como cabecera
    - Elimina filas de pie de página (FUENTE, NOTAS, filas vacías)
    - Parsea la columna de fecha (formato M/D/YYYY)
    - Convierte columnas de valor a numérico
    - Añade columnas de año y mes
    - Opcionalmente descarta columnas (drop_cols)
    """
    df = pd.read_csv(filepath, skiprows=FILAS_METADATA_BDE, encoding='latin1')

    # Eliminar filas de pie de página y vacías
    df = df[df['fecha'].notna()].copy()
    df = df[~df['fecha'].astype(str).str.strip().str.match(r'^(FUENTE|NOTAS|$)')].copy()

    # Descartar columnas no deseadas
    if drop_cols:
        df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    # Parsear fecha (formato M/D/YYYY del BdE)
    df['fecha'] = pd.to_datetime(df['fecha'], format='%m/%d/%Y')

    # Añadir año y mes
    df['año'] = df['fecha'].dt.year
    df['mes'] = df['fecha'].dt.month

    # Convertir columnas de valor a numérico
    cols_valor = [c for c in df.columns if c not in ['fecha', 'año', 'mes']]
    for col in cols_valor:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Reordenar: tiempo primero
    df = df[['fecha', 'año', 'mes'] + cols_valor].sort_values('fecha').reset_index(drop=True)

    print(f'{nombre}:')
    print(f'  Shape: {df.shape} | Rango: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    print(f'  Columnas: {cols_valor}')
    return df


# morosos_discontinuada: sólo 47 obs (2005-2016), sustituida por definición NPL (EBA) desde Circular 4/2017
df_activo = cargar_bde_csv(ARCHIVO_ACTIVO, 'Activo (be0451)', drop_cols=['morosos_discontinuada', 'ed_activo_dudosos'])
df_pasivo = cargar_bde_csv(ARCHIVO_PASIVO, 'Pasivo (be0452)')

Activo (be0451):
  Shape: (771, 14) | Rango: 1962-01-01 → 2026-03-01
  Columnas: ['ed_activo_total', 'ed_activo_creditos_esp', 'ed_activo_credito_sector_publico', 'ed_activo_credito_sector_privado', 'ed_activo_creditos_resto_mundo', 'ed_activo_renta_fija_esp', 'ed_activo_renta_fija_resto_mundo', 'ed_activo_acciones_esp', 'ed_activo_acciones_resto_mundo', 'ed_activo_efectivo', 'ed_activo_otros_no_sectorizados']
Pasivo (be0452):
  Shape: (771, 13) | Rango: 1962-01-01 → 2026-03-01
  Columnas: ['ed_pasivo_total', 'ed_pasivo_depositos_total', 'ed_pasivo_depositos_esp', 'ed_pasivo_depositos_sector_publico', 'ed_pasivo_sector_privado', 'ed_pasivo_deposito_resto_mundo', 'ed_pasivo_deuda_emitida', 'ed_pasivo_patrimonio_ajustes', 'ec_efc_pasivo_obra_social', 'ed_pasivo_periodificacion_otros']


In [5]:
df_activo.tail(3)

,fecha,año,mes,ed_activo_total,ed_activo_creditos_esp,ed_activo_credito_sector_publico,ed_activo_credito_sector_privado,ed_activo_creditos_resto_mundo,ed_activo_renta_fija_esp,ed_activo_renta_fija_resto_mundo,ed_activo_acciones_esp,ed_activo_acciones_resto_mundo,ed_activo_efectivo,ed_activo_otros_no_sectorizados
768,2026-01-01,2026,1,3206534586,287214439,80443723,1173110621,650959619.0,270735235.0,204340505.0,129285700.0,125918749,7095145.0,277430850.0
769,2026-02-01,2026,2,3210767061,241514018,80880142,1176122398,688653159.0,272547292.0,211535552.0,129821336.0,129706207,6771913.0,273215044.0
770,2026-03-01,2026,3,3194134849,220690415,82797554,1184556971,659078174.0,280776925.0,225761445.0,129068012.0,123296811,7700607.0,280407935.0


In [6]:
df_pasivo.tail(3)

,fecha,año,mes,ed_pasivo_total,ed_pasivo_depositos_total,ed_pasivo_depositos_esp,ed_pasivo_depositos_sector_publico,ed_pasivo_sector_privado,ed_pasivo_deposito_resto_mundo,ed_pasivo_deuda_emitida,ed_pasivo_patrimonio_ajustes,ec_efc_pasivo_obra_social,ed_pasivo_periodificacion_otros
768,2026-01-01,2026,1,3.206535e+09,2391328581,95140332,186394787,1566590553,543202909,275174038.0,323476885.0,423268,216131814.0
769,2026-02-01,2026,2,3.210767e+09,2415663629,94061104,183108363,1575950943,562543219,276433206.0,322486519.0,430848,195752859.0
770,2026-03-01,2026,3,3.194135e+09,2395687851,98200525,188746133,1586522952,522218241,274378072.0,313440413.0,440056,210188457.0


# 3. Estructura y correspondencia de columnas

### Activo (be0451)
El **total del activo** es la suma de sus 10 componentes aditivos:

$$\text{ed\_activo\_total} = \text{creditos\_esp} + \text{credito\_sector\_publico} + \text{credito\_sector\_privado} + \text{creditos\_resto\_mundo} + \text{renta\_fija\_esp} + \text{renta\_fija\_resto\_mundo} + \text{acciones\_esp} + \text{acciones\_resto\_mundo} + \text{efectivo} + \text{otros\_residual}$$

La identidad se cumple exactamente en todo el rango 1962-2026 tras el recálculo del residual.

### Pasivo (be0452)
Dos identidades contables:

**Sub-total de depósitos** (disponible desde 1962):
$$\text{ed\_pasivo\_depositos\_total} = \text{depositos\_esp} + \text{depositos\_sector\_publico} + \text{sector\_privado} + \text{deposito\_resto\_mundo}$$

**Total del pasivo** (disponible desde 2005-06):
$$\text{ed\_pasivo\_total} = \text{depositos\_total} + \text{deuda\_emitida} + \text{patrimonio\_ajustes} + \text{obra\_social} + \text{periodificacion\_otros}$$

In [7]:
# ===== ACTIVO =====
ACTIVO_TOTAL = 'ed_activo_total'

# 10 componentes aditivos (total = Σ componentes)
ACTIVO_COMPONENTES = [
    'ed_activo_creditos_esp',
    'ed_activo_credito_sector_publico',
    'ed_activo_credito_sector_privado',
    'ed_activo_creditos_resto_mundo',
    'ed_activo_renta_fija_esp',
    'ed_activo_renta_fija_resto_mundo',
    'ed_activo_acciones_esp',
    'ed_activo_acciones_resto_mundo',
    'ed_activo_efectivo',
    'ed_activo_otros_residual',
]

# ===== PASIVO =====
# Total general del pasivo (disponible desde 2005-06)
PASIVO_TOTAL = 'ed_pasivo_total'

# Sub-total de depósitos (disponible desde 1962)
PASIVO_DEPOSITOS_TOTAL = 'ed_pasivo_depositos_total'

# Componentes de depósitos (depósitos_total = Σ por sector)
PASIVO_COMPONENTES_DEPOSITOS = [
    'ed_pasivo_depositos_esp',
    'ed_pasivo_depositos_sector_publico',
    'ed_pasivo_sector_privado',
    'ed_pasivo_deposito_resto_mundo',
]

# Componentes del pasivo total (pasivo_total = depósitos + estas partidas)
PASIVO_COMPONENTES_TOTAL = [
    'ed_pasivo_depositos_total',
    'ed_pasivo_deuda_emitida',
    'ed_pasivo_patrimonio_ajustes',
    'ec_efc_pasivo_obra_social',
    'ed_pasivo_periodificacion_otros',
]

print(f'Activo: 1 total + {len(ACTIVO_COMPONENTES)} componentes aditivos')
print(f'Pasivo: 1 total general + 1 sub-total depósitos + {len(PASIVO_COMPONENTES_DEPOSITOS)} componentes depósitos + {len(PASIVO_COMPONENTES_TOTAL)} componentes pasivo total')

Activo: 1 total + 10 componentes aditivos
Pasivo: 1 total general + 1 sub-total depósitos + 4 componentes depósitos + 5 componentes pasivo total


In [8]:
# Renombrar: el sufijo _residual documenta que antes de 1979 es un residual calculado
df_activo = df_activo.rename(columns={
    'ed_activo_otros_no_sectorizados': 'ed_activo_otros_residual'
})

# Back-fill de otros_residual para el periodo 1962-1979
# Antes de dic 1979, el BdE no desglosaba esta partida. Se calcula como:
#   total - Σ(otros 9 componentes disponibles, con NaN -> 0)
# En ese periodo el residual absorbe también acciones España (pre-1970),
# efectivo (pre-1970) y renta fija (pre-1980), que aún no se publicaban por separado.
otros_9 = [c for c in ACTIVO_COMPONENTES if c != 'ed_activo_otros_residual']
mask_nan = df_activo['ed_activo_otros_residual'].isna()
df_activo.loc[mask_nan, 'ed_activo_otros_residual'] = (
    df_activo.loc[mask_nan, 'ed_activo_total']
    - df_activo.loc[mask_nan, otros_9].fillna(0).sum(axis=1)
)
print(f'ed_activo_otros_residual: {mask_nan.sum()} valores recalculados como residual (1962-1979)')
print(f'NaN restantes: {df_activo["ed_activo_otros_residual"].isna().sum()}')

ed_activo_otros_residual: 215 valores recalculados como residual (1962-1979)
NaN restantes: 0


# 4. Análisis de valores nulos
Algunas series no están disponibles desde 1962 (especialmente acciones y efectivo en activo, y deuda emitida en pasivo).

In [9]:
def analizar_nulos(df, nombre):
    """Documenta los períodos con valores nulos por columna."""
    print(f'=== {nombre} ===')
    cols_valor = [c for c in df.columns if c not in ['fecha', 'año', 'mes']]
    for col in cols_valor:
        n_nulos = df[col].isna().sum()
        if n_nulos > 0:
            primer_dato = df[df[col].notna()]['fecha'].min()
            ultimo_nulo = df[df[col].isna()]['fecha'].max()
            print(f'ALERTA  {col}: {n_nulos} nulos | primer dato: {primer_dato.date()} | último nulo: {ultimo_nulo.date()}')
        else:
            print(f'CORRECTO  {col}: sin nulos (cobertura completa desde {df["fecha"].min().date()})')
    print()

analizar_nulos(df_activo, 'Activo')
analizar_nulos(df_pasivo, 'Pasivo')

=== Activo ===
CORRECTO  ed_activo_total: sin nulos (cobertura completa desde 1962-01-01)
CORRECTO  ed_activo_creditos_esp: sin nulos (cobertura completa desde 1962-01-01)
CORRECTO  ed_activo_credito_sector_publico: sin nulos (cobertura completa desde 1962-01-01)
CORRECTO  ed_activo_credito_sector_privado: sin nulos (cobertura completa desde 1962-01-01)
ALERTA  ed_activo_creditos_resto_mundo: 1 nulos | primer dato: 1962-01-01 | último nulo: 1968-12-01
ALERTA  ed_activo_renta_fija_esp: 227 nulos | primer dato: 1980-12-01 | último nulo: 1980-11-01
ALERTA  ed_activo_renta_fija_resto_mundo: 227 nulos | primer dato: 1980-12-01 | último nulo: 1980-11-01
ALERTA  ed_activo_acciones_esp: 107 nulos | primer dato: 1970-12-01 | último nulo: 1970-11-01
CORRECTO  ed_activo_acciones_resto_mundo: sin nulos (cobertura completa desde 1962-01-01)
ALERTA  ed_activo_efectivo: 107 nulos | primer dato: 1970-12-01 | último nulo: 1970-11-01
CORRECTO  ed_activo_otros_residual: sin nulos (cobertura completa desd

# 5. Verificación de consistencia

Se comprueba que los totales coinciden con la suma de sus componentes.

**Nota sobre NaN:** Algunas series comienzan después de 1962 (renta fija desde 1980, dudosos desde 1992, etc.). Para la verificación, se rellenan los NaN con 0 en la suma, esto refleja que la partida no existía o no se desglosaba aún. La identidad cuadra exactamente desde la fecha en que todos los componentes están disponibles.

In [10]:
def verificar_mece(df, col_total, cols_componentes, nombre):

    suma = df[cols_componentes].fillna(0).sum(axis=1)
    total = df[col_total]
    diff = (total - suma).abs()

    # Solo verificar en filas donde el total no es NaN
    mask = total.notna()
    diff_valid = diff[mask]

    print(f'{nombre}:')
    print(f'  Observaciones verificadas: {mask.sum()} de {len(df)}')
    print(f'  Diferencia máxima (absoluta): {diff_valid.max():.0f} miles EUR')
    print(f'  Diferencia media (absoluta): {diff_valid.mean():.1f} miles EUR')

    # Filas con diferencia significativa (> 10 miles EUR)
    problemas = df[mask & (diff > 10)][['fecha', col_total]].copy()
    problemas['suma_componentes'] = suma[mask & (diff > 10)]
    problemas['diferencia'] = diff[mask & (diff > 10)]
    if len(problemas) > 0:
        print(f'  {len(problemas)} filas con diferencia > 10 miles EUR:')
        print(problemas.head(10).to_string(index=False))
    else:
        print(f'  Consistencia MECE perfecta (todas las diferencias <= 10 miles EUR)')
    print()


# 1. Activo: total = Σ 10 componentes
verificar_mece(df_activo, ACTIVO_TOTAL, ACTIVO_COMPONENTES,
               'Activo: total vs Σ componentes (10 partidas)')

# 2. Pasivo: depósitos total = Σ depósitos por sector
verificar_mece(df_pasivo, PASIVO_DEPOSITOS_TOTAL, PASIVO_COMPONENTES_DEPOSITOS,
               'Pasivo: depósitos total vs Σ por sector')

# 3. Pasivo: total general = depósitos + deuda + patrimonio + obra social + periodificación
verificar_mece(df_pasivo, PASIVO_TOTAL, PASIVO_COMPONENTES_TOTAL,
               'Pasivo: total general vs Σ componentes (desde 2005-06)')

Activo: total vs Σ componentes (10 partidas):
  Observaciones verificadas: 771 de 771
  Diferencia máxima (absoluta): 3 miles EUR
  Diferencia media (absoluta): 0.4 miles EUR
  Consistencia MECE perfecta (todas las diferencias <= 10 miles EUR)

Pasivo: depósitos total vs Σ por sector:
  Observaciones verificadas: 771 de 771
  Diferencia máxima (absoluta): 2 miles EUR
  Diferencia media (absoluta): 0.2 miles EUR
  Consistencia MECE perfecta (todas las diferencias <= 10 miles EUR)

Pasivo: total general vs Σ componentes (desde 2005-06):
  Observaciones verificadas: 250 de 771
  Diferencia máxima (absoluta): 0 miles EUR
  Diferencia media (absoluta): 0.0 miles EUR
  Consistencia MECE perfecta (todas las diferencias <= 10 miles EUR)



# 6. Conversión de unidades: miles de euros → millones de euros

Para coherencia con los archivos PIB del proyecto (que están en millones de euros), se dividen todos los valores entre 1.000.

$$\text{M EUR} = \text{miles EUR} \times \frac{1}{1000}$$

In [11]:
def convertir_a_millones(df, nombre):
    df_meur = df.copy()
    cols_valor = [c for c in df.columns if c not in ['fecha', 'año', 'mes']]
    df_meur[cols_valor] = df_meur[cols_valor] * FACTOR_MILES_A_MILLONES
    print(f'{nombre}: convertido a millones de euros')
    return df_meur


df_activo_meur = convertir_a_millones(df_activo, 'Activo')
df_pasivo_meur = convertir_a_millones(df_pasivo, 'Pasivo')

# Verificación puntual
print()
print('Verificación — Activo total, enero 1962:')
print(f'  Original: {df_activo.iloc[0]["ed_activo_total"]:,.0f} miles EUR')
print(f'  Convertido: {df_activo_meur.iloc[0]["ed_activo_total"]:,.2f} M EUR')
print()
print('Verificación — Activo total, última observación:')
ult = df_activo.iloc[-1]
ult_m = df_activo_meur.iloc[-1]
print(f'  {ult["fecha"].date()}: {ult["ed_activo_total"]:,.0f} miles EUR → {ult_m["ed_activo_total"]:,.2f} M EUR')

Activo: convertido a millones de euros
Pasivo: convertido a millones de euros

Verificación — Activo total, enero 1962:
  Original: 2,033,194 miles EUR
  Convertido: 2,033.19 M EUR

Verificación — Activo total, última observación:
  2026-03-01: 3,194,134,849 miles EUR → 3,194,134.85 M EUR


# 7. Verificación de cobertura temporal

Ambos archivos deben cubrir el mismo rango mensual sin huecos.

In [12]:
for nombre, df in [('Activo', df_activo_meur), ('Pasivo', df_pasivo_meur)]:
    rango_esperado = pd.date_range(df['fecha'].min(), df['fecha'].max(), freq='MS')
    encontrados = pd.DatetimeIndex(df['fecha'])
    gaps = rango_esperado.difference(encontrados)
    extras = encontrados.difference(rango_esperado)
    print(f'{nombre}:')
    print(f'  Rango: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    print(f'  Observaciones: {len(df)} (esperadas: {len(rango_esperado)})')
    if len(gaps) > 0:
        print(f'ALERTA  {len(gaps)} meses faltantes: {[g.date() for g in gaps[:5]]}...')
    else:
        print(f'CORRECTO  Serie mensual continua, sin huecos')
    print()

# Verificar que ambos cubren el mismo rango
if df_activo_meur['fecha'].min() == df_pasivo_meur['fecha'].min() and \
   df_activo_meur['fecha'].max() == df_pasivo_meur['fecha'].max():
    print('CORRECTO  Activo y Pasivo cubren exactamente el mismo período')
else:
    print('ALERTA  Los rangos de Activo y Pasivo difieren')
    print(f'Activo: {df_activo_meur["fecha"].min().date()} → {df_activo_meur["fecha"].max().date()}')
    print(f'Pasivo: {df_pasivo_meur["fecha"].min().date()} → {df_pasivo_meur["fecha"].max().date()}')

Activo:
  Rango: 1962-01-01 → 2026-03-01
  Observaciones: 771 (esperadas: 771)
CORRECTO  Serie mensual continua, sin huecos

Pasivo:
  Rango: 1962-01-01 → 2026-03-01
  Observaciones: 771 (esperadas: 771)
CORRECTO  Serie mensual continua, sin huecos

CORRECTO  Activo y Pasivo cubren exactamente el mismo período


# 8. Visualización de series principales

In [13]:
# Activo: total y componentes principales (incluyendo renta fija)
fig_act = px.line(
    df_activo_meur.melt('fecha', value_vars=[
        'ed_activo_total', 'ed_activo_credito_sector_privado',
        'ed_activo_creditos_resto_mundo', 'ed_activo_creditos_esp',
        'ed_activo_renta_fija_esp',
    ]),
    x='fecha', y='value', color='variable',
    title='Entidades de Depósito — Activo: total, créditos principales y renta fija (M EUR)',
    labels={'value': 'Millones EUR', 'fecha': '', 'variable': 'Serie'}
)
fig_act.show()

In [14]:
# Pasivo: total depósitos y componentes por sector
fig_pas = px.line(
    df_pasivo_meur.melt('fecha', value_vars=[
        'ed_pasivo_depositos_total', 'ed_pasivo_sector_privado',
        'ed_pasivo_deposito_resto_mundo', 'ed_pasivo_depositos_esp'
    ]),
    x='fecha', y='value', color='variable',
    title='Entidades de Depósito — Pasivo: depósitos por sector (M EUR)',
    labels={'value': 'Millones EUR', 'fecha': '', 'variable': 'Serie'}
)
fig_pas.show()

In [15]:
# Crédito al sector privado vs Depósitos del sector privado
df_priv = pd.DataFrame({
    'fecha': df_activo_meur['fecha'],
    'credito_sector_privado': df_activo_meur['ed_activo_credito_sector_privado'],
    'depositos_sector_privado': df_pasivo_meur['ed_pasivo_sector_privado'],
})
df_priv['ratio_credito_depositos'] = df_priv['credito_sector_privado'] / df_priv['depositos_sector_privado']

fig_ratio = px.line(
    df_priv, x='fecha', y='ratio_credito_depositos',
    title='Ratio crédito / depósitos del sector privado (OSR)',
    labels={'ratio_credito_depositos': 'Ratio', 'fecha': ''}
)
fig_ratio.show()

# 9. Resumen de los datasets finales

In [16]:
for nombre, df in [('Activo', df_activo_meur), ('Pasivo', df_pasivo_meur)]:
    cols_val = [c for c in df.columns if c not in ['fecha', 'año', 'mes']]
    print(f'=== {nombre} ===')
    print(f'  Shape: {df.shape}')
    print(f'  Rango: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    print(f'  Unidad: millones de euros')
    print(f'  Frecuencia: mensual')
    print(f'  Columnas de valor: {cols_val}')
    print(f'  Nulos por columna:')
    for col in cols_val:
        n = df[col].isna().sum()
        if n > 0:
            primer = df[df[col].notna()]['fecha'].min().date()
            print(f'    {col}: {n} nulos (disponible desde {primer})')
        else:
            print(f'    {col}: 0 nulos')
    print()

=== Activo ===
  Shape: (771, 14)
  Rango: 1962-01-01 → 2026-03-01
  Unidad: millones de euros
  Frecuencia: mensual
  Columnas de valor: ['ed_activo_total', 'ed_activo_creditos_esp', 'ed_activo_credito_sector_publico', 'ed_activo_credito_sector_privado', 'ed_activo_creditos_resto_mundo', 'ed_activo_renta_fija_esp', 'ed_activo_renta_fija_resto_mundo', 'ed_activo_acciones_esp', 'ed_activo_acciones_resto_mundo', 'ed_activo_efectivo', 'ed_activo_otros_residual']
  Nulos por columna:
    ed_activo_total: 0 nulos
    ed_activo_creditos_esp: 0 nulos
    ed_activo_credito_sector_publico: 0 nulos
    ed_activo_credito_sector_privado: 0 nulos
    ed_activo_creditos_resto_mundo: 1 nulos (disponible desde 1962-01-01)
    ed_activo_renta_fija_esp: 227 nulos (disponible desde 1980-12-01)
    ed_activo_renta_fija_resto_mundo: 227 nulos (disponible desde 1980-12-01)
    ed_activo_acciones_esp: 107 nulos (disponible desde 1970-12-01)
    ed_activo_acciones_resto_mundo: 0 nulos
    ed_activo_efectivo: 

# 10. Exportación de archivos finales

- `ed_activo_historico_1962_2026.csv`
- `ed_pasivo_historico_1962_2026.csv`

**Unidad:** millones de euros. **Frecuencia:** mensual.

In [17]:
# ── Recortar al rango de tasa de paro y guardar en Datasets ────────────
from pathlib import Path
FECHA_INICIO = '1974-07-01'
FECHA_FIN    = '2025-10-01'
ruta_datasets = Path(r'C:\Users\marco\PycharmProjects\TFM_Marcos\Datasets')

# Orden canónico de columnas: total → componentes aditivos
time_cols = ['fecha', 'año', 'mes']

activo_val_cols = [
    # TOTAL
    'ed_activo_total',
    # Componentes aditivos (total = Σ de estos 10)
    'ed_activo_creditos_esp',
    'ed_activo_credito_sector_publico',
    'ed_activo_credito_sector_privado',
    'ed_activo_creditos_resto_mundo',
    'ed_activo_renta_fija_esp',
    'ed_activo_renta_fija_resto_mundo',
    'ed_activo_acciones_esp',
    'ed_activo_acciones_resto_mundo',
    'ed_activo_efectivo',
    'ed_activo_otros_residual',
]

pasivo_val_cols = [
    # TOTAL GENERAL (desde 2005-06)
    'ed_pasivo_total',
    # SUB-TOTAL DEPÓSITOS + sus 4 componentes sectoriales
    'ed_pasivo_depositos_total',
    'ed_pasivo_depositos_esp',
    'ed_pasivo_depositos_sector_publico',
    'ed_pasivo_sector_privado',
    'ed_pasivo_deposito_resto_mundo',
    # Otras partidas del pasivo
    'ed_pasivo_deuda_emitida',
    'ed_pasivo_patrimonio_ajustes',
    'ec_efc_pasivo_obra_social',
    'ed_pasivo_periodificacion_otros',
]

df_activo_final = df_activo_meur[time_cols + activo_val_cols]
df_pasivo_final = df_pasivo_meur[time_cols + pasivo_val_cols]

# ── Deflactar a precios constantes 2025 ────────────────────────────────────
df_ipc = pd.read_csv(ruta_datasets / 'indice_precios_consumo_IPC_diferentes_bases.csv')
df_ipc['fecha'] = pd.to_datetime(df_ipc['fecha'])
deflactor = df_ipc.set_index('fecha')['IPC_2025']

for nombre, df_bal in [('activo', df_activo_final), ('pasivo', df_pasivo_final)]:
    df_bal = df_bal.copy()
    df_bal['fecha_dt'] = pd.to_datetime(df_bal['fecha'])
    df_bal = df_bal.merge(deflactor, left_on='fecha_dt', right_index=True, how='left')
    val_cols = [c for c in df_bal.columns
                if c.startswith(('ed_', 'ec_')) and df_bal[c].dtype in ('float64', 'int64')]
    for col in val_cols:
        df_bal[col] = df_bal[col] / (df_bal['IPC_2025'] / 100)
    df_bal.drop(columns=['fecha_dt', 'IPC_2025'], inplace=True)
    if nombre == 'activo':
        df_activo_final = df_bal
    else:
        df_pasivo_final = df_bal

print(f'Deflactado con IPC_2025: activo {df_activo_final.shape}, pasivo {df_pasivo_final.shape}')


df_activo_final = df_activo_final[(df_activo_final['fecha'] >= FECHA_INICIO) & (df_activo_final['fecha'] <= FECHA_FIN)].copy()
df_activo_final.to_csv(ruta_datasets / 'ed_activo_historico_1962_2026.csv', index=False, encoding='utf-8-sig')
df_pasivo_final = df_pasivo_final[(df_pasivo_final['fecha'] >= FECHA_INICIO) & (df_pasivo_final['fecha'] <= FECHA_FIN)].copy()
df_pasivo_final.to_csv(ruta_datasets / 'ed_pasivo_historico_1962_2026.csv', index=False, encoding='utf-8-sig')

print(f'Activo exportado: {df_activo_final.shape} columnas: {activo_val_cols}')
print(f'Pasivo exportado: {df_pasivo_final.shape} columnas: {pasivo_val_cols}')

Deflactado con IPC_2025: activo (771, 14), pasivo (771, 13)
Activo exportado: (616, 14) columnas: ['ed_activo_total', 'ed_activo_creditos_esp', 'ed_activo_credito_sector_publico', 'ed_activo_credito_sector_privado', 'ed_activo_creditos_resto_mundo', 'ed_activo_renta_fija_esp', 'ed_activo_renta_fija_resto_mundo', 'ed_activo_acciones_esp', 'ed_activo_acciones_resto_mundo', 'ed_activo_efectivo', 'ed_activo_otros_residual']
Pasivo exportado: (616, 13) columnas: ['ed_pasivo_total', 'ed_pasivo_depositos_total', 'ed_pasivo_depositos_esp', 'ed_pasivo_depositos_sector_publico', 'ed_pasivo_sector_privado', 'ed_pasivo_deposito_resto_mundo', 'ed_pasivo_deuda_emitida', 'ed_pasivo_patrimonio_ajustes', 'ec_efc_pasivo_obra_social', 'ed_pasivo_periodificacion_otros']


# 11. Información acerca de los datasets finales

In [18]:
# Información de los datasets finales exportados
for nombre, df in [('ACTIVO', df_activo_final), ('PASIVO', df_pasivo_final)]:
    print(f'Información acerca de {nombre}:')
    print(f'{df.shape[0]} meses × {df.shape[1]} columnas | Rango: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    columnas = df.columns.tolist()
    print(f'  Columnas: {columnas}')
    print()
    nulos = df.isna().sum()
    nulos_col = nulos[nulos > 0]
    if len(nulos_col) > 0:
        print(f'ALERTA  {nombre}: nulos en {nulos_col.to_dict()}')
    else:
        print(f'CORRECTO  {nombre}: sin valores nulos')
    print()
    print(f'=== {nombre} (primeras y últimas 2 filas) ===')
    print(pd.concat([df.head(2), df.tail(2)]).to_string(index=False))
    print()

Información acerca de ACTIVO:
616 meses × 14 columnas | Rango: 1974-07-01 → 2025-10-01
  Columnas: ['fecha', 'año', 'mes', 'ed_activo_total', 'ed_activo_creditos_esp', 'ed_activo_credito_sector_publico', 'ed_activo_credito_sector_privado', 'ed_activo_creditos_resto_mundo', 'ed_activo_renta_fija_esp', 'ed_activo_renta_fija_resto_mundo', 'ed_activo_acciones_esp', 'ed_activo_acciones_resto_mundo', 'ed_activo_efectivo', 'ed_activo_otros_residual']

ALERTA  ACTIVO: nulos en {'ed_activo_renta_fija_esp': 77, 'ed_activo_renta_fija_resto_mundo': 77}

=== ACTIVO (primeras y últimas 2 filas) ===
     fecha  año  mes  ed_activo_total  ed_activo_creditos_esp  ed_activo_credito_sector_publico  ed_activo_credito_sector_privado  ed_activo_creditos_resto_mundo  ed_activo_renta_fija_esp  ed_activo_renta_fija_resto_mundo  ed_activo_acciones_esp  ed_activo_acciones_resto_mundo  ed_activo_efectivo  ed_activo_otros_residual
1974-07-01 1974    7     4.589941e+05            62652.167628                       

In [19]:
for nombre, df in [('ACTIVO', df_activo_final), ('PASIVO', df_pasivo_final)]:
    print(f'=== {nombre} ===')
    print(df.describe().round(2))
    print()

=== ACTIVO ===
                               fecha      año     mes  ed_activo_total  \
count                            616   616.00  616.00           616.00   
mean   2000-02-15 07:59:13.246753280  1999.67    6.51       2081551.02   
min              1974-07-01 00:00:00  1974.00    1.00        445449.78   
25%              1987-04-23 12:00:00  1987.00    4.00        853855.47   
50%              2000-02-15 12:00:00  2000.00    7.00       1744962.97   
75%              2012-12-08 18:00:00  2012.25    9.25       3200210.42   
max              2025-10-01 00:00:00  2025.00   12.00       4327079.85   
std                              NaN    14.83    3.45       1273246.86   

       ed_activo_creditos_esp  ed_activo_credito_sector_publico  \
count                  616.00                            616.00   
mean                221898.37                          52317.14   
min                  58332.84                            319.99   
25%                 165786.98                     

In [20]:
for nombre, df in [('ACTIVO', df_activo_final), ('PASIVO', df_pasivo_final)]:
    print(f'=== {nombre} ===')
    df.info()
    print()

=== ACTIVO ===
<class 'pandas.core.frame.DataFrame'>
Index: 616 entries, 150 to 765
Data columns (total 14 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   fecha                             616 non-null    datetime64[ns]
 1   año                               616 non-null    int32         
 2   mes                               616 non-null    int32         
 3   ed_activo_total                   616 non-null    float64       
 4   ed_activo_creditos_esp            616 non-null    float64       
 5   ed_activo_credito_sector_publico  616 non-null    float64       
 6   ed_activo_credito_sector_privado  616 non-null    float64       
 7   ed_activo_creditos_resto_mundo    616 non-null    float64       
 8   ed_activo_renta_fija_esp          539 non-null    float64       
 9   ed_activo_renta_fija_resto_mundo  539 non-null    float64       
 10  ed_activo_acciones_esp            616 